# Завдання 4. CNN для MNIST: прості та глибші архітектури, k-fold, збереження

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import KFold

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = (X_train.reshape(-1,28,28,1) / 255.0).astype('float32')   # [ширина][висота][канали], channels last
X_test  = (X_test.reshape(-1,28,28,1)  / 255.0).astype('float32')
y_train_c, y_test_c = to_categorical(y_train), to_categorical(y_test)
num_classes = 10

def plot_history(h, title=''):
    fig, ax = plt.subplots(1,2, figsize=(11,3.5))
    ax[0].plot(h.history['accuracy'], label='train'); ax[0].plot(h.history['val_accuracy'], label='val'); ax[0].set_title(title+' accuracy'); ax[0].legend()
    ax[1].plot(h.history['loss'], label='train'); ax[1].plot(h.history['val_loss'], label='val'); ax[1].set_title(title+' loss'); ax[1].legend()
    plt.show()

## Модель 1. Проста CNN

In [ ]:
def cnn_simple():
    m = keras.Sequential([
        layers.Input(shape=(28,28,1)),
        layers.Conv2D(32, (5,5), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    m.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return m

m1 = cnn_simple()
h1 = m1.fit(X_train, y_train_c, validation_data=(X_test, y_test_c), epochs=10, batch_size=200, verbose=2)
s = m1.evaluate(X_test, y_test_c, verbose=0)
print("Модель 1: точність %.2f%%, помилка CNN: %.2f%%" % (s[1]*100, 100-s[1]*100))
plot_history(h1, 'CNN simple')

## Модель 2. Ускладнена CNN

In [ ]:
def cnn_medium():
    m = keras.Sequential([
        layers.Input(shape=(28,28,1)),
        layers.Conv2D(30, (5,5), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(15, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(50, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    m.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return m

m2 = cnn_medium()
h2 = m2.fit(X_train, y_train_c, validation_data=(X_test, y_test_c), epochs=10, batch_size=200, verbose=2)
s = m2.evaluate(X_test, y_test_c, verbose=0)
print("Модель 2: точність %.2f%%, помилка: %.2f%%" % (s[1]*100, 100-s[1]*100))
plot_history(h2, 'CNN medium')

## Модель 3. Глибша CNN (VGG-подібна) + k-fold (k=5)

In [ ]:
def cnn_deep():
    m = keras.Sequential([
        layers.Input(shape=(28,28,1)),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    m.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return m

def evaluate_kfold(build_fn, k=5, epochs=10, batch=64):
    kf = KFold(n_splits=k, shuffle=True, random_state=1)
    scores, histories = [], []
    for fold, (tr, va) in enumerate(kf.split(X_train), 1):
        es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        m = build_fn()
        h = m.fit(X_train[tr], y_train_c[tr], validation_data=(X_train[va], y_train_c[va]),
                  epochs=epochs, batch_size=batch, callbacks=[es], verbose=0)
        acc = m.evaluate(X_train[va], y_train_c[va], verbose=0)[1]
        print(f"Fold {fold}: {acc*100:.2f}%")
        scores.append(acc); histories.append(h)
    print("Середня точність: %.3f%% (std=%.3f)  n=%d" % (np.mean(scores)*100, np.std(scores)*100, len(scores)))
    return scores, histories

scores, hists = evaluate_kfold(cnn_deep)
plt.boxplot(scores); plt.title('5-fold CV, deep CNN'); plt.show()

## Остаточна модель: навчання на всьому train, збереження

In [ ]:
final = cnn_deep()
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
hf = final.fit(X_train, y_train_c, validation_split=0.1, epochs=20, batch_size=64, callbacks=[es], verbose=2)
plot_history(hf, 'Final deep CNN')
s = final.evaluate(X_test, y_test_c, verbose=0)
print("Тест: точність %.2f%%, помилка %.2f%%" % (s[1]*100, 100-s[1]*100))
final.save('final_cnn_mnist.keras')

In [ ]:
# Перевірка 10 прикладів з датасету зі збереженої моделі
loaded = keras.models.load_model('final_cnn_mnist.keras')
pred = loaded.predict(X_test[:10], verbose=0).argmax(1)
plt.figure(figsize=(12,2.5))
for i in range(10):
    plt.subplot(1,10,i+1); plt.imshow(X_test[i].squeeze(), cmap='gray'); plt.axis('off')
    plt.title(f"{pred[i]} ({'✓' if pred[i]==y_test[i] else '✗'})")
plt.show()
print("Правильно:", (pred==y_test[:10]).sum(), "з 10")

## Підсумок для звіту (заповніть таблицю своїми результатами)
| Модель | Test accuracy | Помилка | Перенавчання? |
|---|---|---|---|
| MLP (завд. 2) | | | |
| CNN simple | | | |
| CNN medium | | | |
| CNN deep + k-fold | | | |

Проти перенавчання: Dropout, BatchNorm, EarlyStopping, k-fold, валідаційна вибірка.